# Norway Olympics Medals vs Norway Market — Comparable Event Study Notebook

This notebook is designed to mirror the structure of the USA workflow as closely as possible, but for **Norway**.

It does all four steps in one place:

1. **Filters Norway Olympic medals** from the raw Olympics medal dataset
2. **Scrapes event dates** for each unique medal event and drops medals where a specific date cannot be recovered
3. **Maps medal dates to Oslo trading days** and downloads Norwegian market data for the event window and full panel
4. **Runs the same style of analysis as the USA notebook** and exports result CSVs

## Outputs written by this notebook

- `event_dates_lookup.csv`
- `olympic_medals_with_event_dates.csv`
- `medal_daily_counts_and_score_ALL.csv`
- `medal_daily_counts_and_score_SUMMER.csv`
- `medal_daily_counts_and_score_WINTER.csv`
- `medals_market_panel.csv`
- `results_corr_matrix.csv`
- `results_diff_in_means.csv`
- `results_core_regressions_long.csv`
- `results_sector_regressions_long.csv`
- `results_season_interaction_long.csv`
- `results_lead_lag_long.csv`
- `results_car_summary.csv`

## Notes on comparability

- The **Olympics side** is kept the same as the USA notebook: same medal weighting, same event-date recovery logic, same mapping to the next trading day, same regression/test families.
- The **market side** is adapted to Norway: Oslo exchange calendar, Norwegian market benchmark, and Norway-listed equities.
- For abnormal returns, the notebook uses a **rolling market model with OSEBX as the benchmark**. That preserves the event-study logic used in the USA analysis while using a benchmark that is appropriate for Norway.


In [24]:
%pip -q install pandas numpy requests beautifulsoup4 lxml tqdm python-dateutil rapidfuzz yfinance pandas_market_calendars statsmodels scipy

Note: you may need to restart the kernel to use updated packages.


In [25]:
import warnings
warnings.filterwarnings("ignore")

import re
import time
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any, List

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import pandas_market_calendars as mcal
import statsmodels.api as sm

from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from dateutil import parser as dtparser
from rapidfuzz import fuzz
from scipy import stats
from statsmodels.regression.rolling import RollingOLS

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# ---------- paths ----------
CACHE_DIR = Path("../cache/cache_wiki_norway")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = Path("../main/dataset/olympic_medals.csv")

# ---------- analysis controls ----------
FILTER_COUNTRY_3 = "NOR"
FILTER_MEDAL_TYPE = "ALL"   # ALL / GOLD / SILVER / BRONZE
YEAR_MIN, YEAR_MAX = 2000, 2024
MEDAL_WEIGHTS = {"GOLD": 3, "SILVER": 2, "BRONZE": 1}

# Broad market + concentrated Norway-listed names
MARKET_TICKER = "OSEBX.OL"
TICKERS = ["OSEBX.OL", "EQNR.OL", "DNB.OL", "KOG.OL", "MOWI.OL", "ORK.OL"]
TICKER_LABELS = {
    "OSEBX.OL": "OSEBX",
    "EQNR.OL": "EQNR",
    "DNB.OL": "DNB",
    "KOG.OL": "KOG",
    "MOWI.OL": "MOWI",
    "ORK.OL": "ORK",
}
ROLLING_WINDOW = 120
EVENT_WINDOW_DAYS = 5

# ---------- scraping session ----------
USER_AGENT = "OlympicsMarketEventStudyBot/1.0 (educational notebook)"
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})
WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKI_REST = "https://en.wikipedia.org/api/rest_v1/page/html"
WIKIDATA_API = "https://www.wikidata.org/w/api.php"


## 1) Load the raw medal dataset and filter to Norway

In [26]:
df = pd.read_csv(RAW_PATH)
print("Raw shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

cols_clean = ["discipline_title", "slug_game", "event_title", "event_gender", "medal_type", "country_3_letter_code"]
for c in cols_clean:
    df[c] = df[c].astype(str).str.strip()

df_nor = df[df["country_3_letter_code"] == FILTER_COUNTRY_3].copy()
if FILTER_MEDAL_TYPE.upper() != "ALL":
    df_nor = df_nor[df_nor["medal_type"] == FILTER_MEDAL_TYPE].copy()

df_nor["year"] = pd.to_numeric(df_nor["slug_game"].str.extract(r"(\d{4})")[0], errors="coerce")
df_nor = df_nor.dropna(subset=["year"]).copy()
df_nor["year"] = df_nor["year"].astype(int)
df_nor = df_nor[(df_nor["year"] >= YEAR_MIN) & (df_nor["year"] <= YEAR_MAX)].copy()

# Same summer/winter assignment used in the USA workflow for 2000–2024
# Summer Olympics years are divisible by 4; Winter Olympics years in this span are not.
df_nor["season"] = np.where(df_nor["year"] % 4 == 0, "S", "W")

MEDAL_KEY = ["slug_game", "discipline_title", "event_title", "event_gender", "medal_type", "country_3_letter_code"]
EVENT_KEY = ["slug_game", "discipline_title", "event_title", "event_gender"]

df_nor_dedup_medals = df_nor.drop_duplicates(subset=MEDAL_KEY).copy()
events_nor = df_nor_dedup_medals[EVENT_KEY].drop_duplicates().reset_index(drop=True)

print("Filtered Norway rows:", df_nor.shape)
print("Unique medal awards after dedupe:", df_nor_dedup_medals.shape[0])
print("Unique Norway events to scrape:", len(events_nor))
display(events_nor.head(10))


Raw shape: (21697, 12)
Columns: ['discipline_title', 'slug_game', 'event_title', 'event_gender', 'medal_type', 'participant_type', 'participant_title', 'athlete_url', 'athlete_full_name', 'country_name', 'country_code', 'country_3_letter_code']


,discipline_title,slug_game,event_title,event_gender,medal_type,participant_type,participant_title,athlete_url,athlete_full_name,country_name,country_code,country_3_letter_code
0,Curling,beijing-2022,Mixed Doubles,Mixed,GOLD,GameTeam,Italy,https://olympics.com/en/athletes/stefania-cons...,Stefania CONSTANTINI,Italy,IT,ITA
1,Curling,beijing-2022,Mixed Doubles,Mixed,GOLD,GameTeam,Italy,https://olympics.com/en/athletes/amos-mosaner,Amos MOSANER,Italy,IT,ITA
2,Curling,beijing-2022,Mixed Doubles,Mixed,SILVER,GameTeam,Norway,https://olympics.com/en/athletes/kristin-skaslien,Kristin SKASLIEN,Norway,NO,NOR
3,Curling,beijing-2022,Mixed Doubles,Mixed,SILVER,GameTeam,Norway,https://olympics.com/en/athletes/magnus-nedreg...,Magnus NEDREGOTTEN,Norway,NO,NOR
4,Curling,beijing-2022,Mixed Doubles,Mixed,BRONZE,GameTeam,Sweden,https://olympics.com/en/athletes/almida-de-val,Almida DE VAL,Sweden,SE,SWE


Filtered Norway rows: (223, 14)
Unique medal awards after dedupe: 209
Unique Norway events to scrape: 188


,slug_game,discipline_title,event_title,event_gender
0,beijing-2022,Curling,Mixed Doubles,Mixed
1,beijing-2022,Freestyle Skiing,Men's Freeski Big Air,Men
2,beijing-2022,Snowboard,Men's Snowboard Big Air,Men
3,beijing-2022,Ski Jumping,Men's LH Individual,Men
4,beijing-2022,Biathlon,Men's 12.5km Pursuit,Men
5,beijing-2022,Biathlon,Mixed Relay 4x6km (W+M),Mixed
6,beijing-2022,Biathlon,Women's 12.5km Mass Start,Women
7,beijing-2022,Biathlon,Women's 10km Pursuit,Women
8,beijing-2022,Biathlon,Men's 10km Sprint,Men
9,beijing-2022,Biathlon,Women's 7.5km Sprint,Women


## 2) Scraping helpers for Olympic event dates

In [27]:
def slug_to_year(slug_game: str) -> Optional[int]:
    m = re.search(r"(\d{4})", str(slug_game))
    return int(m.group(1)) if m else None


def wiki_search(query: str, limit: int = 8) -> List[Dict[str, Any]]:
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "srlimit": limit,
        "format": "json",
        "utf8": 1,
    }
    r = SESSION.get(WIKI_API, params=params, timeout=30)
    r.raise_for_status()
    return r.json().get("query", {}).get("search", [])


def choose_best_wiki_title(results: List[Dict[str, Any]], discipline: str, event: str, year: int) -> Optional[str]:
    if not results:
        return None
    target = f"{discipline} {event} {year} olympics".lower()
    scored = []
    for it in results:
        title = it.get("title", "")
        t = title.lower()
        score = 0
        if str(year) in t:
            score += 20
        if "olympic" in t:
            score += 15
        score += int(fuzz.token_set_ratio(t, target) * 0.5)
        scored.append((score, title))
    scored.sort(reverse=True, key=lambda x: x[0])
    return scored[0][1] if scored else None


def cache_read(path: Path) -> Optional[str]:
    if path.exists():
        return path.read_text(encoding="utf-8", errors="ignore")
    return None


def cache_write(path: Path, text: str) -> None:
    path.write_text(text, encoding="utf-8")


def wiki_html(title: str) -> str:
    safe = re.sub(r"[^a-zA-Z0-9._-]+", "_", title)[:200]
    cache_path = CACHE_DIR / f"html__{safe}.html"
    cached = cache_read(cache_path)
    if cached is not None:
        return cached

    url = f"{WIKI_REST}/{title.replace(' ', '_')}"
    r = SESSION.get(url, timeout=30)
    r.raise_for_status()
    cache_write(cache_path, r.text)
    return r.text


def wiki_wikitext(title: str) -> str:
    safe = re.sub(r"[^a-zA-Z0-9._-]+", "_", title)[:200]
    cache_path = CACHE_DIR / f"wikitext__{safe}.txt"
    cached = cache_read(cache_path)
    if cached is not None:
        return cached

    params = {
        "action": "query",
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "format": "json",
        "titles": title,
        "redirects": 1,
    }
    r = SESSION.get(WIKI_API, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    pages = data.get("query", {}).get("pages", {})
    text = ""
    for _, page in pages.items():
        revs = page.get("revisions", [])
        if not revs:
            continue
        slots = revs[0].get("slots", {})
        text = slots.get("main", {}).get("*", "") or ""
    cache_write(cache_path, text)
    return text


def extract_infobox_dates_from_html(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    infobox = soup.find("table", class_=re.compile(r"infobox"))
    if not infobox:
        return None
    for row in infobox.find_all("tr"):
        th = row.find("th")
        td = row.find("td")
        if not th or not td:
            continue
        key = th.get_text(" ", strip=True).lower()
        if key in ("dates", "date"):
            return td.get_text(" ", strip=True)
    return None


def extract_infobox_dates_from_wikitext(wt: str) -> Optional[str]:
    if not wt:
        return None

    for key in ["dates", "date"]:
        m = re.search(rf"\|\s*{key}\s*=\s*([^\n\|]+)", wt, flags=re.IGNORECASE)
        if m:
            val = m.group(1).strip()
            val = re.sub(r"\[\[([^\]|]+)(\|[^\]]+)?\]\]", r"\1", val)
            val = re.sub(r"\{\{[^\}]+\}\}", "", val)
            val = re.sub(r"<[^>]+>", "", val).strip()
            return val if val else None

    return None


def wiki_title_to_wikidata_qid(title: str) -> Optional[str]:
    params = {
        "action": "query",
        "prop": "pageprops",
        "titles": title,
        "format": "json",
        "redirects": 1,
    }
    r = SESSION.get(WIKI_API, params=params, timeout=30)
    r.raise_for_status()
    pages = r.json().get("query", {}).get("pages", {})
    for _, page in pages.items():
        qid = page.get("pageprops", {}).get("wikibase_item")
        if qid:
            return qid
    return None


def wikidata_get_event_date_end(qid: str, year: int) -> Optional[str]:
    params = {
        "action": "wbgetentities",
        "ids": qid,
        "props": "claims",
        "format": "json",
    }
    r = SESSION.get(WIKIDATA_API, params=params, timeout=30)
    r.raise_for_status()
    data = r.json().get("entities", {}).get(qid, {})
    claims = data.get("claims", {})
    for prop in ["P585", "P582", "P580"]:
        for claim in claims.get(prop, []):
            mainsnak = claim.get("mainsnak", {})
            dv = mainsnak.get("datavalue", {})
            val = dv.get("value", {})
            timestr = val.get("time")
            if not timestr:
                continue
            m = re.match(r"^[+\-]?(\d{4})-(\d{2})-(\d{2})", timestr)
            if not m:
                continue
            y, mo, da = map(int, m.groups())
            if y == year:
                return f"{y:04d}-{mo:02d}-{da:02d}"
    return None


def parse_event_date_end(datestr: str, year: int) -> Optional[str]:
    if not datestr or not year:
        return None

    s = str(datestr)
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"\[[^\]]+\]", "", s)
    s = re.sub(r"\([^\)]*\)", "", s).strip()

    # try full parse directly
    try:
        dt = dtparser.parse(s, fuzzy=True, default=datetime(year, 1, 1))
        if dt.year == year:
            return dt.strftime("%Y-%m-%d")
    except Exception:
        pass

    # range forms like "29 July - 1 August 2021"
    range_match = re.search(r"(.+?)\s*-\s*(.+)", s)
    if range_match:
        rhs = range_match.group(2).strip()
        try:
            dt = dtparser.parse(rhs, fuzzy=True, default=datetime(year, 1, 1))
            if dt.year == year:
                return dt.strftime("%Y-%m-%d")
        except Exception:
            pass

    # compact range like "12-15 February 2002"
    compact = re.search(r"(\d{1,2})\s*-\s*(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})", s)
    if compact:
        d2 = compact.group(2)
        mo = compact.group(3)
        yr = int(compact.group(4))
        try:
            dt = dtparser.parse(f"{d2} {mo} {yr}", fuzzy=True)
            return dt.strftime("%Y-%m-%d")
        except Exception:
            pass

    return None


def scrape_event_date(slug_game: str, discipline: str, event: str, gender: str) -> Dict[str, Any]:
    year = slug_to_year(slug_game)
    if not year:
        return {"wiki_title": None, "raw_dates": None, "event_date_end": None, "status": "no_year_in_slug"}

    queries = [
        f'"{event}" {discipline} {year} Olympics',
        f"{discipline} at the {year} Olympics {event}",
        f'"{event}" {year} Olympic {discipline}',
        f"{discipline} {event} {year} Olympic",
    ]

    wiki_title = None
    search_cache_key = re.sub(r"[^a-zA-Z0-9._-]+", "_", f"{discipline}__{event}__{year}")[:180]
    search_cache_path = CACHE_DIR / f"search__{search_cache_key}.json"
    if search_cache_path.exists():
        cached = pd.read_json(search_cache_path, typ="series").to_dict()
        wiki_title = cached.get("wiki_title")

    if not wiki_title:
        for q in queries:
            try:
                res = wiki_search(q, limit=10)
            except Exception:
                continue
            title = choose_best_wiki_title(res, discipline, event, year)
            if title:
                wiki_title = title
                break
        search_cache_path.write_text(pd.Series({"wiki_title": wiki_title}).to_json(), encoding="utf-8")

    if not wiki_title:
        return {"wiki_title": None, "raw_dates": None, "event_date_end": None, "status": "no_wiki_match"}

    try:
        html = wiki_html(wiki_title)
        raw_dates = extract_infobox_dates_from_html(html)
        if raw_dates:
            iso = parse_event_date_end(raw_dates, year)
            if iso:
                return {"wiki_title": wiki_title, "raw_dates": raw_dates, "event_date_end": iso, "status": "ok_infobox"}
            out = {"wiki_title": wiki_title, "raw_dates": raw_dates, "event_date_end": None, "status": "date_parse_failed"}
        else:
            out = {"wiki_title": wiki_title, "raw_dates": None, "event_date_end": None, "status": "no_infobox_dates"}
    except Exception as e:
        return {"wiki_title": wiki_title, "raw_dates": None, "event_date_end": None, "status": f"fetch_failed_html: {e}"}

    try:
        qid = wiki_title_to_wikidata_qid(wiki_title)
        if qid:
            wd = wikidata_get_event_date_end(qid, year)
            if wd:
                return {"wiki_title": wiki_title, "raw_dates": "Wikidata", "event_date_end": wd, "status": "ok_wikidata", "wikidata_qid": qid}
    except Exception as e:
        out["wikidata_error"] = str(e)

    try:
        wt = wiki_wikitext(wiki_title)
        wt_dates = extract_infobox_dates_from_wikitext(wt)
        if wt_dates:
            iso = parse_event_date_end(wt_dates, year)
            if iso:
                return {"wiki_title": wiki_title, "raw_dates": wt_dates, "event_date_end": iso, "status": "ok_wikitext"}
            out["wikitext_raw_dates"] = wt_dates
            out["status"] = "date_parse_failed_wikitext"
        else:
            out["wikitext_raw_dates"] = None
            if out["status"] == "no_infobox_dates":
                out["status"] = "no_dates_infobox_or_wikitext"
    except Exception as e:
        out["wikitext_error"] = str(e)

    return out


## 3) Run the Norway event-date scrape, drop unresolved events, and build daily medal counts

In [28]:
# Small sample first
sample = events_nor.sample(min(20, len(events_nor)), random_state=42).reset_index(drop=True)
rows = []
for _, r in tqdm(sample.iterrows(), total=len(sample)):
    out = scrape_event_date(
        slug_game=r["slug_game"],
        discipline=r["discipline_title"],
        event=r["event_title"],
        gender=r["event_gender"],
    )
    rows.append({**r.to_dict(), **out})
    time.sleep(0.5)

sample_dates = pd.DataFrame(rows)
display(sample_dates.head(20))
print(sample_dates["status"].value_counts())


100%|██████████| 20/20 [00:15<00:00,  1.30it/s]


,slug_game,discipline_title,event_title,event_gender,wiki_title,raw_dates,event_date_end,status,wikitext_raw_dates,wikidata_qid
0,sydney-2000,Sailing,fleetmatch race keelboat open Soling mixed,Open,NaT,None,None,fetch_failed_html: expected string or bytes-li...,NaN,NaN
1,salt-lake-city-2002,Alpine Skiing,downhill men,Men,Alpine skiing at the 2002 Winter Olympics – Wo...,"February 12, 2002",2002-02-12,ok_infobox,NaN,NaN
2,beijing-2022,Cross Country Skiing,Men's Team Sprint Classic,Men,2023–24 FIS Cross-Country World Cup,None,None,date_parse_failed_wikitext,15 February 2023,NaN
3,beijing-2022,Alpine Skiing,Men's Super-G,Men,Alpine skiing at the 2022 Winter Olympics – Me...,8 February 2022,2022-02-08,ok_infobox,NaN,NaN
4,pyeongchang-2018,Cross Country Skiing,Ladies’ 10km Free,Women,North Korea at the 2018 Winter Olympics,Wikidata,2018-01-01,ok_wikidata,NaN,Q41510590
5,vancouver-2010,Alpine Skiing,giant slalom men,Men,Alpine skiing at the 2010 Winter Olympics,15–27 February 2010,2010-02-27,ok_infobox,NaN,NaN
6,pyeongchang-2018,Biathlon,Men’s 20km Individual,Men,Chronological summary of the 2018 Winter Olympics,Wikidata,2018-00-00,ok_wikidata,NaN,Q48179800
7,rio-2016,Wrestling,Greco-Roman 59 kg men,Men,Wrestling at the 2016 Summer Olympics – Men's ...,14 August 2016,2016-08-14,ok_infobox,NaN,NaN
8,beijing-2022,Alpine Skiing,Mixed Team Parallel,Mixed,Alpine skiing at the 2022 Winter Olympics – Mi...,20 February,2022-02-20,ok_infobox,NaN,NaN
9,turin-2006,Ski Jumping,Normal Hill Individual men,Men,Ski jumping at the Winter Olympics,None,None,date_parse_failed_wikitext,11 February 2014,NaN


status
ok_infobox                                                                                                                                                                                    7
date_parse_failed_wikitext                                                                                                                                                                    4
ok_wikidata                                                                                                                                                                                   4
fetch_failed_html: expected string or bytes-like object, got 'NaTType'                                                                                                                        2
no_dates_infobox_or_wikitext                                                                                                                                                                  1
ok_wikitext                      

In [29]:
OUT_PATH = Path("../data/interim/norway/norway_event_dates_lookup.csv")

if OUT_PATH.exists():
    event_dates = pd.read_csv(OUT_PATH)
    print("Loaded existing Norway lookup:", event_dates.shape)
else:
    event_dates = pd.DataFrame(columns=list(events_nor.columns) + [
        "wiki_title", "raw_dates", "event_date_end", "status", "wikidata_qid", "wikidata_error", "wikitext_raw_dates", "wikitext_error"
    ])

done_keys = set(tuple(x) for x in event_dates[EVENT_KEY].values.tolist()) if len(event_dates) else set()

to_do = events_nor[~events_nor.apply(lambda r: tuple(r[k] for k in EVENT_KEY) in done_keys, axis=1)]
print("Remaining Norway events:", len(to_do))

rows = []
for _, r in tqdm(to_do.iterrows(), total=len(to_do)):
    out = scrape_event_date(
        slug_game=r["slug_game"],
        discipline=r["discipline_title"],
        event=r["event_title"],
        gender=r["event_gender"],
    )
    rows.append({**r.to_dict(), **out})
    time.sleep(0.5)
    if len(rows) >= 100:
        event_dates = pd.concat([event_dates, pd.DataFrame(rows)], ignore_index=True)
        event_dates.to_csv(OUT_PATH, index=False)
        rows = []

if rows:
    event_dates = pd.concat([event_dates, pd.DataFrame(rows)], ignore_index=True)
    event_dates.to_csv(OUT_PATH, index=False)

print("Lookup table saved:", event_dates.shape)
display(event_dates.head())
print(event_dates["status"].value_counts())


Loaded existing Norway lookup: (188, 12)
Remaining Norway events: 0


0it [00:00, ?it/s]

Lookup table saved: (188, 12)


,slug_game,discipline_title,event_title,event_gender,wiki_title,raw_dates,event_date_end,status,wikidata_qid,wikidata_error,wikitext_raw_dates,wikitext_error
0,beijing-2022,Curling,Mixed Doubles,Mixed,Curling at the 2022 Winter Olympics – Mixed do...,2–8 February,2022-02-08,ok_infobox,NaN,NaN,NaN,NaN
1,beijing-2022,Freestyle Skiing,Men's Freeski Big Air,Men,New Zealand at the 2022 Winter Olympics,Wikidata,2022-01-01,ok_wikidata,Q48784914,NaN,NaN,NaN
2,beijing-2022,Snowboard,Men's Snowboard Big Air,Men,New Zealand at the 2022 Winter Olympics,Wikidata,2022-01-01,ok_wikidata,Q48784914,NaN,NaN,NaN
3,beijing-2022,Ski Jumping,Men's LH Individual,Men,Ski jumping at the 2010 Winter Olympics – Larg...,19–20 February 2010,2010-02-20,ok_infobox,NaN,NaN,NaN,NaN
4,beijing-2022,Biathlon,Men's 12.5km Pursuit,Men,Biathlon at the 2026 Winter Olympics – Men's p...,15 February 2026,NaN,date_parse_failed_wikitext,NaN,NaN,15 February 2026,NaN


status
no_wiki_match                                                                                                                                                                                                              42
ok_infobox                                                                                                                                                                                                                 31
date_parse_failed_wikitext                                                                                                                                                                                                 21
ok_wikidata                                                                                                                                                                                                                21
ok_wikitext                                                                                              

In [ ]:
missing_events = event_dates[event_dates["event_date_end"].isna()].copy()
event_dates_clean = event_dates[event_dates["event_date_end"].notna()].copy()

print("Unique events missing dates:", len(missing_events))
print("Remaining unique events after drop:", len(event_dates_clean))
if len(event_dates):
    print("Dropped %:", round(100 * len(missing_events) / len(event_dates), 2), "%")
display(missing_events.head())

# Merge dates back to Norway rows

df_nor_enriched = df_nor.merge(
    event_dates_clean[EVENT_KEY + ["event_date_end", "raw_dates", "wiki_title", "status", "wikidata_qid"]],
    on=EVENT_KEY,
    how="left",
)

# Daily aggregation exactly like USA workflow

df_nor_enriched["event_date_end"] = pd.to_datetime(df_nor_enriched["event_date_end"], errors="coerce")
df_medals = df_nor_enriched.drop_duplicates(subset=MEDAL_KEY).copy()
valid_medals = {"GOLD", "SILVER", "BRONZE"}
df_medals = df_medals[df_medals["medal_type"].isin(valid_medals)].copy()
df_medals["medal_points"] = df_medals["medal_type"].map(MEDAL_WEIGHTS).astype(float)

df_medals = df_medals[
    (df_medals["event_date_end"].dt.year >= YEAR_MIN) &
    (df_medals["event_date_end"].dt.year <= YEAR_MAX)
].copy()

norway_medal_daily_all = (
    df_medals.groupby(["event_date_end", "season"])
    .agg(
        total_medals=("medal_type", "size"),
        gold=("medal_type", lambda s: (s == "GOLD").sum()),
        silver=("medal_type", lambda s: (s == "SILVER").sum()),
        bronze=("medal_type", lambda s: (s == "BRONZE").sum()),
        performance_score=("medal_points", "sum"),
    )
    .reset_index()
    .sort_values("event_date_end")
)

cols = ["event_date_end", "total_medals", "gold", "silver", "bronze", "performance_score", "season"]
norway_medal_daily_all = norway_medal_daily_all[cols]
norway_medal_daily_summer = norway_medal_daily_all[norway_medal_daily_all["season"] == "S"].copy()
norway_medal_daily_winter = norway_medal_daily_all[norway_medal_daily_all["season"] == "W"].copy()

norway_medal_daily_all.to_csv("../outputs/norway_medal_daily_counts_and_score_ALL.csv", index=False)
norway_medal_daily_summer.to_csv("../outputs/norway_medal_daily_counts_and_score_SUMMER.csv", index=False)
norway_medal_daily_winter.to_csv("../outputs/norway_medal_daily_counts_and_score_WINTER.csv", index=False)

print("ALL:", norway_medal_daily_all.shape)
print("SUMMER:", norway_medal_daily_summer.shape)
print("WINTER:", norway_medal_daily_winter.shape)
display(norway_medal_daily_all.head())

Unique events missing dates: 125
Remaining unique events after drop: 63
Dropped %: 66.49 %


,slug_game,discipline_title,event_title,event_gender,wiki_title,raw_dates,event_date_end,status,wikidata_qid,wikidata_error,wikitext_raw_dates,wikitext_error
4,beijing-2022,Biathlon,Men's 12.5km Pursuit,Men,Biathlon at the 2026 Winter Olympics – Men's p...,15 February 2026,NaN,date_parse_failed_wikitext,NaN,NaN,15 February 2026,NaN
5,beijing-2022,Biathlon,Mixed Relay 4x6km (W+M),Mixed,2021–22 in skiing,NaN,NaN,no_dates_infobox_or_wikitext,NaN,NaN,NaN,NaN
6,beijing-2022,Biathlon,Women's 12.5km Mass Start,Women,Biathlon at the 2026 Winter Olympics – Women's...,21 February 2026,NaN,date_parse_failed_wikitext,NaN,NaN,21 February 2026,NaN
7,beijing-2022,Biathlon,Women's 10km Pursuit,Women,Biathlon at the 2026 Winter Olympics – Women's...,15 February 2026,NaN,date_parse_failed_wikitext,NaN,NaN,15 February 2026,NaN
9,beijing-2022,Biathlon,Women's 7.5km Sprint,Women,Biathlon at the 2026 Winter Olympics – Women's...,14 February 2026,NaN,date_parse_failed_wikitext,NaN,NaN,14 February 2026,NaN


Wrote: norway_olympic_medals_with_event_dates.csv
ALL: (36, 7)
SUMMER: (5, 7)
WINTER: (31, 7)


,event_date_end,total_medals,gold,silver,bronze,performance_score,season
0,2002-02-11,1,0,1,0,2.0,W
1,2002-02-12,1,0,1,0,2.0,W
2,2002-02-20,1,0,0,1,1.0,W
3,2010-01-01,1,0,1,0,2.0,W
4,2010-02-20,1,1,0,0,3.0,W


## 4) Map medal days to Oslo trading days and build the full trading-day panel

In [31]:
medals = pd.read_csv("../data/interim/norway/norway_medal_daily_counts_and_score_ALL.csv")
medals = medals.rename(columns={"event_date_end": "medal_date"})
medals["medal_date"] = pd.to_datetime(medals["medal_date"]).dt.normalize()

# Try Oslo calendars in a robust order
cal = None
for cal_name in ["XOSL", "OSE"]:
    try:
        cal = mcal.get_calendar(cal_name)
        print("Using calendar:", cal_name)
        break
    except Exception:
        pass
if cal is None:
    raise RuntimeError("Could not load an Oslo Stock Exchange calendar from pandas_market_calendars.")

start = medals["medal_date"].min() - pd.Timedelta(days=15)
end = medals["medal_date"].max() + pd.Timedelta(days=15)
schedule = cal.schedule(start_date=start, end_date=end)
trading_days = pd.DatetimeIndex(schedule.index).normalize()

def to_next_trading_day(d: pd.Timestamp) -> pd.Timestamp:
    d = pd.Timestamp(d).normalize()
    idx = trading_days.searchsorted(d)
    if idx >= len(trading_days):
        return pd.NaT
    return trading_days[idx]

medals["trade_date"] = medals["medal_date"].map(to_next_trading_day)
medals = medals.dropna(subset=["trade_date"]).copy()
medals["trade_date"] = pd.to_datetime(medals["trade_date"]).dt.normalize()
medals["winter_dummy"] = (medals["season"] == "W").astype(int)
medals["any_medal_dummy"] = (medals["total_medals"] > 0).astype(int)

full_schedule = cal.schedule(start_date=medals["trade_date"].min(), end_date=medals["trade_date"].max())
full_trading_days = pd.DatetimeIndex(full_schedule.index).normalize()

panel = pd.DataFrame({"trade_date": full_trading_days})
panel = panel.merge(
    medals[[
        "trade_date", "season", "gold", "silver", "bronze", "total_medals",
        "performance_score", "winter_dummy", "any_medal_dummy"
    ]],
    on="trade_date",
    how="left",
)

fill_zero = ["gold", "silver", "bronze", "total_medals", "performance_score", "winter_dummy", "any_medal_dummy"]
for c in fill_zero:
    panel[c] = panel[c].fillna(0)
panel["season"] = panel["season"].fillna("N")

print("Medal-day rows:", medals.shape)
print("Full trading-day panel:", panel.shape)
display(panel.head())


Using calendar: XOSL
Medal-day rows: (36, 10)
Full trading-day panel: (5099, 9)


,trade_date,season,gold,silver,bronze,total_medals,performance_score,winter_dummy,any_medal_dummy
0,2002-02-11,W,0.0,1.0,0.0,1.0,2.0,1.0,1.0
1,2002-02-12,W,0.0,1.0,0.0,1.0,2.0,1.0,1.0
2,2002-02-13,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2002-02-14,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2002-02-15,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5) Download Norwegian market data and compute returns, volumes, volatility, and abnormal returns

In [32]:
market_start = panel["trade_date"].min() - pd.Timedelta(days=400)
market_end = panel["trade_date"].max() + pd.Timedelta(days=EVENT_WINDOW_DAYS + 5)

raw = yf.download(
    TICKERS,
    start=market_start.strftime("%Y-%m-%d"),
    end=(market_end + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    auto_adjust=False,
    progress=False,
    group_by="ticker",
)

if raw.empty:
    raise RuntimeError("Yahoo Finance returned no data for the requested Norway tickers.")

rows = []
for t in TICKERS:
    if isinstance(raw.columns, pd.MultiIndex):
        if t not in raw.columns.get_level_values(0):
            print(f"Skipping missing ticker: {t}")
            continue
        sub = raw[t].copy()
    else:
        sub = raw.copy()
    sub = sub.reset_index().rename(columns={"Date": "date"})
    sub.columns = [str(c) for c in sub.columns]
    if "Adj Close" in sub.columns:
        px_col = "Adj Close"
    elif "Close" in sub.columns:
        px_col = "Close"
    else:
        continue
    keep = [c for c in ["date", px_col, "Volume"] if c in sub.columns]
    sub = sub[keep].copy()
    sub = sub.rename(columns={px_col: "price", "Volume": "volume"})
    sub["date"] = pd.to_datetime(sub["date"]).dt.normalize()
    sub["ticker"] = t
    rows.append(sub)

market_long = pd.concat(rows, ignore_index=True).sort_values(["ticker", "date"]).reset_index(drop=True)
market_long["ret"] = market_long.groupby("ticker")["price"].transform(lambda s: np.log(s).diff())
market_long["lvc"] = market_long.groupby("ticker")["volume"].transform(lambda s: np.log(s.replace(0, np.nan)).diff())
market_long["vol20"] = market_long.groupby("ticker")["ret"].transform(lambda s: s.rolling(20, min_periods=10).std())

# Use OSEBX as the benchmark market series
mkt = market_long[market_long["ticker"] == MARKET_TICKER][["date", "ret"]].rename(columns={"ret": "mkt_ret"}).copy()
market_long = market_long.merge(mkt, on="date", how="left")


def compute_market_model_abret(df_t: pd.DataFrame, window: int = 120) -> pd.DataFrame:
    x = sm.add_constant(df_t[["mkt_ret"]], has_constant="add")
    y = df_t["ret"]
    valid = x.notna().all(axis=1) & y.notna()
    out = df_t.copy()
    out["exp_ret_capm"] = np.nan
    out["abret"] = np.nan

    if valid.sum() < window:
        return out

    model = RollingOLS(y[valid], x[valid], window=window)
    res = model.fit()
    params = res.params.reindex(df_t.index)
    out.loc[params.index, "exp_ret_capm"] = (
        params["const"] + params["mkt_ret"] * out.loc[params.index, "mkt_ret"]
    )
    out.loc[params.index, "abret"] = out.loc[params.index, "ret"] - out.loc[params.index, "exp_ret_capm"]
    return out


def compute_index_abret(df_t: pd.DataFrame, window: int = 120) -> pd.DataFrame:
    out = df_t.copy()
    out["exp_ret_capm"] = out["ret"].rolling(window, min_periods=window).mean()
    out["abret"] = out["ret"] - out["exp_ret_capm"]
    return out

parts = []
for t, g in market_long.groupby("ticker", sort=False):
    g = g.sort_values("date").copy()
    if t == MARKET_TICKER:
        g = compute_index_abret(g, window=ROLLING_WINDOW)
    else:
        g = compute_market_model_abret(g, window=ROLLING_WINDOW)
    parts.append(g)
market_long = pd.concat(parts, ignore_index=True)

display(market_long.head())


,date,price,volume,ticker,ret,lvc,vol20,mkt_ret,exp_ret_capm,abret
0,2001-01-08,17.044373,552465.0,DNB.OL,NaN,NaN,NaN,NaN,NaN,NaN
1,2001-01-09,16.791597,603368.0,DNB.OL,-0.014941,0.088137,NaN,NaN,NaN,NaN
2,2001-01-10,16.719372,1018185.0,DNB.OL,-0.004311,0.523250,NaN,NaN,NaN,NaN
3,2001-01-11,16.611042,732479.0,DNB.OL,-0.006500,-0.329342,NaN,NaN,NaN,NaN
4,2001-01-12,16.141600,2013702.0,DNB.OL,-0.028668,1.011295,NaN,NaN,NaN,NaN


In [33]:
# Build a wide table for panel merge
market_long["label"] = market_long["ticker"].map(TICKER_LABELS)
market_long["ret_col"] = "ret_" + market_long["label"]
market_long["lvc_col"] = "lvc_" + market_long["label"]
market_long["absret_col"] = "absret_" + market_long["label"]
market_long["vol20_col"] = "vol20_" + market_long["label"]

ret_w = market_long.pivot(index="date", columns="ret_col", values="ret")
lvc_w = market_long.pivot(index="date", columns="lvc_col", values="lvc")
ab_w = market_long.pivot(index="date", columns="absret_col", values="abret")
vol_w = market_long.pivot(index="date", columns="vol20_col", values="vol20")

wide = pd.concat([ret_w, lvc_w, ab_w, vol_w], axis=1).reset_index().rename(columns={"date": "trade_date"})
panel = panel.merge(wide, on="trade_date", how="left")

panel.to_csv("../data/interim/norway/norway_medals_market_panel.csv", index=False)
print("Wrote: norway_medals_market_panel.csv")
display(panel.head())


Wrote: norway_medals_market_panel.csv


,trade_date,season,gold,silver,bronze,total_medals,performance_score,winter_dummy,any_medal_dummy,ret_DNB,ret_EQNR,ret_KOG,ret_MOWI,ret_ORK,ret_OSEBX,lvc_DNB,lvc_EQNR,lvc_KOG,lvc_MOWI,lvc_ORK,lvc_OSEBX,absret_DNB,absret_EQNR,absret_KOG,absret_MOWI,absret_ORK,absret_OSEBX,vol20_DNB,vol20_EQNR,vol20_KOG,vol20_MOWI,vol20_ORK,vol20_OSEBX
0,2002-02-11,W,0.0,1.0,0.0,1.0,2.0,1.0,1.0,0.012330,0.008584,-0.009390,0.039806,-0.006472,NaN,-0.018533,-0.286239,-4.823422,0.593397,-0.698132,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.014135,0.016891,0.011817,0.030237,0.013024,NaN
1,2002-02-12,W,0.0,1.0,0.0,1.0,2.0,1.0,1.0,0.004890,0.000000,-0.014252,-0.034743,-0.003252,NaN,-0.004473,-0.181101,NaN,-0.779448,1.429347,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.014070,0.016784,0.012327,0.028289,0.013045,NaN
2,2002-02-13,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.021713,0.033617,0.014252,-0.010152,0.000000,NaN,0.372152,0.388612,NaN,0.001494,-1.949836,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.014171,0.018238,0.012690,0.024963,0.012399,NaN
3,2002-02-14,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.030555,0.024491,0.000000,-0.010257,-0.013114,NaN,1.283908,0.300965,0.023247,-0.962882,1.263094,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.015283,0.018628,0.012422,0.022283,0.012007,NaN
4,2002-02-15,N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,-0.009479,-0.042111,-0.009950,NaN,-1.820014,-0.268596,-0.510869,1.843932,-0.060617,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.015183,0.018582,0.011951,0.024584,0.011330,NaN


## 6) Regression helpers and analysis outputs

In [34]:
def add_calendar_controls(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["trade_date"] = pd.to_datetime(out["trade_date"])
    out["dow"] = out["trade_date"].dt.dayofweek
    out["month"] = out["trade_date"].dt.month

    dow_d = pd.get_dummies(out["dow"], prefix="dow", drop_first=True, dtype=int)
    mo_d = pd.get_dummies(out["month"], prefix="month", drop_first=True, dtype=int)
    out = pd.concat([out, dow_d, mo_d], axis=1)
    return out


def run_ols(df: pd.DataFrame, y: str, xvars: List[str]):
    cols = [y] + xvars
    d = df[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(d) < max(25, len(xvars) + 5):
        return None
    X = sm.add_constant(d[xvars], has_constant="add")
    model = sm.OLS(d[y], X).fit(cov_type="HC1")
    return model


def tidy_model(model, model_name: str, y_name: str) -> pd.DataFrame:
    if model is None:
        return pd.DataFrame(columns=["model", "y", "term", "coef", "std_err", "t", "p", "nobs", "r2"])
    return pd.DataFrame({
        "model": model_name,
        "y": y_name,
        "term": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "t": model.tvalues.values,
        "p": model.pvalues.values,
        "nobs": model.nobs,
        "r2": model.rsquared,
    })


def diff_means(df: pd.DataFrame, col: str):
    a = df.loc[df["any_medal_dummy"] == 1, col].dropna()
    b = df.loc[df["any_medal_dummy"] == 0, col].dropna()
    if len(a) < 3 or len(b) < 3:
        return {"metric": col, "mean_medal": np.nan, "mean_nonmedal": np.nan, "diff": np.nan, "t": np.nan, "p": np.nan}
    t, p = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")
    return {"metric": col, "mean_medal": a.mean(), "mean_nonmedal": b.mean(), "diff": a.mean() - b.mean(), "t": t, "p": p}


def compute_car(df: pd.DataFrame, ar_col: str, event_dates: pd.Series, window=(-1, 1)) -> pd.Series:
    idx_map = {d: i for i, d in enumerate(df["trade_date"])}
    vals = []
    for d in event_dates.dropna().unique():
        i = idx_map.get(pd.Timestamp(d))
        if i is None:
            continue
        lo = i + window[0]
        hi = i + window[1]
        if lo < 0 or hi >= len(df):
            continue
        vals.append(df.loc[lo:hi, ar_col].sum())
    return pd.Series(vals, name=f"CAR_{window[0]}_{window[1]}")


In [35]:
dfc = add_calendar_controls(panel)
control_cols = [c for c in dfc.columns if c.startswith("dow_") or c.startswith("month_")]

summary_cols = [
    "gold", "silver", "bronze", "total_medals", "performance_score",
    "any_medal_dummy", "winter_dummy",
    "ret_OSEBX", "lvc_OSEBX", "absret_OSEBX", "vol20_OSEBX",
]
summary_cols = [c for c in summary_cols if c in dfc.columns]
summary = dfc[summary_cols].describe().T
corr = dfc[summary_cols].corr(numeric_only=True)

summary.to_csv("../results/norway/results_summary_stats.csv")
corr.to_csv("../results/norway/results_corr_matrix.csv")
print("Wrote summary + correlation CSVs")
display(summary)
display(corr)


Wrote summary + correlation CSVs


,count,mean,std,min,25%,50%,75%,max
gold,5099.0,0.003530,0.068522,0.000000,0.000000,0.000000,0.000000,3.000000
silver,5099.0,0.003922,0.068501,0.000000,0.000000,0.000000,0.000000,2.000000
bronze,5099.0,0.003922,0.074007,0.000000,0.000000,0.000000,0.000000,2.000000
total_medals,5099.0,0.011375,0.164134,0.000000,0.000000,0.000000,0.000000,6.000000
performance_score,5099.0,0.022357,0.330677,0.000000,0.000000,0.000000,0.000000,13.000000
any_medal_dummy,5099.0,0.007060,0.083736,0.000000,0.000000,0.000000,0.000000,1.000000
winter_dummy,5099.0,0.006080,0.077742,0.000000,0.000000,0.000000,0.000000,1.000000
ret_OSEBX,2282.0,0.000445,0.010669,-0.091755,-0.004713,0.000701,0.006237,0.054560
lvc_OSEBX,1055.0,-0.005798,0.387333,-4.679188,-0.186090,-0.003791,0.160449,4.315807
absret_OSEBX,1455.0,0.000121,0.012040,-0.089064,-0.005642,0.000456,0.006406,0.056878


,gold,silver,bronze,total_medals,performance_score,any_medal_dummy,winter_dummy,ret_OSEBX,lvc_OSEBX,absret_OSEBX,vol20_OSEBX
gold,1.000000,0.373162,0.461444,0.781274,0.879528,0.611016,0.585131,0.028832,-0.006420,0.017280,0.001900
silver,0.373162,1.000000,0.383897,0.746227,0.732201,0.679118,0.658533,0.024726,0.031558,0.010261,0.007168
bronze,0.461444,0.383897,1.000000,0.803750,0.669712,0.628594,0.609540,0.002173,0.026271,-0.004660,-0.000580
total_medals,0.781274,0.746227,0.803750,1.000000,0.974729,0.821938,0.793949,0.022803,0.019850,0.009401,0.003283
performance_score,0.879528,0.732201,0.669712,0.974729,1.000000,0.801884,0.772999,0.028395,0.011757,0.014137,0.003821
any_medal_dummy,0.611016,0.679118,0.628594,0.821938,0.801884,1.000000,0.927503,0.026479,0.011119,0.005805,0.006675
winter_dummy,0.585131,0.658533,0.609540,0.793949,0.772999,0.927503,1.000000,0.026517,0.005797,0.004381,0.007897
ret_OSEBX,0.028832,0.024726,0.002173,0.022803,0.028395,0.026479,0.026517,1.000000,-0.031340,0.998104,0.028609
lvc_OSEBX,-0.006420,0.031558,0.026271,0.019850,0.011757,0.011119,0.005797,-0.031340,1.000000,0.002981,0.006352
absret_OSEBX,0.017280,0.010261,-0.004660,0.009401,0.014137,0.005805,0.004381,0.998104,0.002981,1.000000,0.075769


In [36]:
metrics = [c for c in ["ret_OSEBX", "lvc_OSEBX", "absret_OSEBX", "vol20_OSEBX"] if c in dfc.columns]
diff_tbl = pd.DataFrame([diff_means(dfc, c) for c in metrics])
diff_tbl.to_csv("../results/norway/results_diff_in_means.csv", index=False)
print("Wrote: results_diff_in_means.csv")
display(diff_tbl)


Wrote: results_diff_in_means.csv


,metric,mean_medal,mean_nonmedal,diff,t,p
0,ret_OSEBX,0.002934,0.000413,0.002522,1.897630,0.067509
1,lvc_OSEBX,0.032743,-0.006279,0.039022,0.460066,0.653370
2,absret_OSEBX,0.000784,0.000114,0.000670,0.340962,0.737619
3,vol20_OSEBX,0.009862,0.009586,0.000276,0.625631,0.536149


In [37]:
core_rows = []
Y_LIST = [c for c in ["ret_OSEBX", "lvc_OSEBX", "absret_OSEBX", "vol20_OSEBX"] if c in dfc.columns]
X_SETS = {
    "any_only": ["any_medal_dummy"] + control_cols,
    "score_only": ["performance_score"] + control_cols,
    "full_medal_mix": ["gold", "silver", "bronze"] + control_cols,
}
for y in Y_LIST:
    for name, xvars in X_SETS.items():
        m = run_ols(dfc, y, xvars)
        core_rows.append(tidy_model(m, f"core::{name}", y))
core = pd.concat(core_rows, ignore_index=True)
core.to_csv("../results/norway/results_core_regressions_long.csv", index=False)
print("Wrote: results_core_regressions_long.csv")
display(core.head(20))


Wrote: results_core_regressions_long.csv


,model,y,term,coef,std_err,t,p,nobs,r2
0,core::any_only,ret_OSEBX,const,-0.000948,0.000944,-1.004126,0.315318,2282.0,0.009897
1,core::any_only,ret_OSEBX,any_medal_dummy,0.002816,0.001455,1.936192,0.052844,2282.0,0.009897
2,core::any_only,ret_OSEBX,dow_1,0.001451,0.000737,1.967900,0.049080,2282.0,0.009897
3,core::any_only,ret_OSEBX,dow_2,0.001397,0.000733,1.906433,0.056594,2282.0,0.009897
4,core::any_only,ret_OSEBX,dow_3,-0.000586,0.000789,-0.741845,0.458182,2282.0,0.009897
5,core::any_only,ret_OSEBX,dow_4,0.000810,0.000744,1.088807,0.276239,2282.0,0.009897
6,core::any_only,ret_OSEBX,month_2,0.000720,0.001235,0.582571,0.560182,2282.0,0.009897
7,core::any_only,ret_OSEBX,month_3,-0.000037,0.001324,-0.028044,0.977627,2282.0,0.009897
8,core::any_only,ret_OSEBX,month_4,0.002030,0.001085,1.870401,0.061428,2282.0,0.009897
9,core::any_only,ret_OSEBX,month_5,0.001513,0.001070,1.413698,0.157451,2282.0,0.009897


In [38]:
sec_rows = []
for label in ["EQNR", "DNB", "KOG", "MOWI", "ORK"]:
    for y in [f"ret_{label}", f"absret_{label}"]:
        if y not in dfc.columns:
            continue
        for name, xvars in X_SETS.items():
            m = run_ols(dfc, y, xvars)
            sec_rows.append(tidy_model(m, f"{label}::{name}", y))
sector = pd.concat(sec_rows, ignore_index=True) if sec_rows else pd.DataFrame()
sector.to_csv("../results/norway/results_sector_regressions_long.csv", index=False)
print("Wrote: results_sector_regressions_long.csv")
display(sector.head(20))


Wrote: results_sector_regressions_long.csv


,model,y,term,coef,std_err,t,p,nobs,r2
0,EQNR::any_only,ret_EQNR,const,-0.000296,0.001033,-0.286988,0.774121,5099.0,0.002855
1,EQNR::any_only,ret_EQNR,any_medal_dummy,0.006585,0.002531,2.601564,0.009280,5099.0,0.002855
2,EQNR::any_only,ret_EQNR,dow_1,0.000664,0.000861,0.771644,0.440325,5099.0,0.002855
3,EQNR::any_only,ret_EQNR,dow_2,0.000210,0.000853,0.246056,0.805639,5099.0,0.002855
4,EQNR::any_only,ret_EQNR,dow_3,-0.000193,0.000884,-0.218361,0.827148,5099.0,0.002855
5,EQNR::any_only,ret_EQNR,dow_4,0.000220,0.000871,0.252003,0.801039,5099.0,0.002855
6,EQNR::any_only,ret_EQNR,month_2,0.001682,0.001313,1.281235,0.200111,5099.0,0.002855
7,EQNR::any_only,ret_EQNR,month_3,0.000813,0.001345,0.604678,0.545393,5099.0,0.002855
8,EQNR::any_only,ret_EQNR,month_4,0.002457,0.001288,1.907756,0.056423,5099.0,0.002855
9,EQNR::any_only,ret_EQNR,month_5,0.000636,0.001296,0.490869,0.623519,5099.0,0.002855


In [39]:
df_si = dfc.copy()
df_si["score_x_winter"] = df_si["performance_score"] * df_si["winter_dummy"]
si_rows = []
for y in Y_LIST:
    m = run_ols(df_si, y, ["performance_score", "winter_dummy", "score_x_winter"] + control_cols)
    si_rows.append(tidy_model(m, "season_interaction", y))
season_interaction = pd.concat(si_rows, ignore_index=True)
season_interaction.to_csv("../results/norway/results_season_interaction_long.csv", index=False)
print("Wrote: results_season_interaction_long.csv")
display(season_interaction.head(20))


Wrote: results_season_interaction_long.csv


,model,y,term,coef,std_err,t,p,nobs,r2
0,season_interaction,ret_OSEBX,const,-0.000994,0.000949,-1.047447,0.294893,2282.0,0.010125
1,season_interaction,ret_OSEBX,performance_score,0.000858,0.000833,1.029558,0.303218,2282.0,0.010125
2,season_interaction,ret_OSEBX,winter_dummy,0.001283,0.002092,0.612957,0.539905,2282.0,0.010125
3,season_interaction,ret_OSEBX,score_x_winter,-0.000337,0.000878,-0.384244,0.700798,2282.0,0.010125
4,season_interaction,ret_OSEBX,dow_1,0.001455,0.000738,1.970401,0.048792,2282.0,0.010125
5,season_interaction,ret_OSEBX,dow_2,0.001408,0.000734,1.919256,0.054952,2282.0,0.010125
6,season_interaction,ret_OSEBX,dow_3,-0.000581,0.000790,-0.735685,0.461923,2282.0,0.010125
7,season_interaction,ret_OSEBX,dow_4,0.000820,0.000744,1.101235,0.270795,2282.0,0.010125
8,season_interaction,ret_OSEBX,month_2,0.000769,0.001256,0.612264,0.540363,2282.0,0.010125
9,season_interaction,ret_OSEBX,month_3,0.000003,0.001327,0.002166,0.998272,2282.0,0.010125


In [40]:
df_ll = dfc.copy()
for v in ["performance_score", "any_medal_dummy", "gold", "silver", "bronze"]:
    if v in df_ll.columns:
        df_ll[f"{v}_lag1"] = df_ll[v].shift(1)
        df_ll[f"{v}_lead1"] = df_ll[v].shift(-1)

ll_rows = []
for y in Y_LIST:
    xvars = [c for c in ["performance_score_lag1", "performance_score", "performance_score_lead1"] if c in df_ll.columns] + control_cols
    m = run_ols(df_ll, y, xvars)
    ll_rows.append(tidy_model(m, "lead_lag_score", y))
lead_lag = pd.concat(ll_rows, ignore_index=True)
lead_lag.to_csv("../results/norway/results_lead_lag_long.csv", index=False)
print("Wrote: results_lead_lag_long.csv")
display(lead_lag.head(20))


Wrote: results_lead_lag_long.csv


,model,y,term,coef,std_err,t,p,nobs,r2
0,lead_lag_score,ret_OSEBX,const,-0.001065,0.000950,-1.121415,0.262111,2281.0,0.010973
1,lead_lag_score,ret_OSEBX,performance_score_lag1,0.000187,0.000224,0.833094,0.404792,2281.0,0.010973
2,lead_lag_score,ret_OSEBX,performance_score,0.000823,0.000262,3.138182,0.001700,2281.0,0.010973
3,lead_lag_score,ret_OSEBX,performance_score_lead1,0.000544,0.000317,1.713380,0.086643,2281.0,0.010973
4,lead_lag_score,ret_OSEBX,dow_1,0.001454,0.000738,1.970420,0.048790,2281.0,0.010973
5,lead_lag_score,ret_OSEBX,dow_2,0.001445,0.000732,1.973837,0.048400,2281.0,0.010973
6,lead_lag_score,ret_OSEBX,dow_3,-0.000584,0.000789,-0.740778,0.458828,2281.0,0.010973
7,lead_lag_score,ret_OSEBX,dow_4,0.000799,0.000744,1.075041,0.282356,2281.0,0.010973
8,lead_lag_score,ret_OSEBX,month_2,0.000707,0.001218,0.580507,0.561573,2281.0,0.010973
9,lead_lag_score,ret_OSEBX,month_3,0.000071,0.001329,0.053791,0.957102,2281.0,0.010973


In [41]:
car_rows = []
event_dates = dfc.loc[dfc["any_medal_dummy"] == 1, "trade_date"]
for label in ["OSEBX", "EQNR", "DNB", "KOG", "MOWI", "ORK"]:
    ar_col = f"absret_{label}"
    if ar_col not in dfc.columns:
        continue
    for win in [(-1, 1), (0, 1), (0, 3), (0, 5)]:
        s = compute_car(dfc.reset_index(drop=True), ar_col, event_dates, window=win)
        car_rows.append({
            "ticker": label,
            "window": f"{win[0]} to {win[1]}",
            "n_events": int(s.notna().sum()),
            "mean_car": s.mean(),
            "median_car": s.median(),
            "std_car": s.std(),
            "t_stat_vs_0": (s.mean() / (s.std(ddof=1) / np.sqrt(len(s)))) if len(s) > 1 and s.std(ddof=1) > 0 else np.nan,
        })
car = pd.DataFrame(car_rows)
car.to_csv("../results/norway/results_car_summary.csv", index=False)
print("Wrote: results_car_summary.csv")
display(car)


Wrote: results_car_summary.csv


,ticker,window,n_events,mean_car,median_car,std_car,t_stat_vs_0
0,OSEBX,-1 to 1,31,0.003465,0.000000,0.008302,2.323862
1,OSEBX,0 to 1,32,0.001751,0.000000,0.005341,1.854582
2,OSEBX,0 to 3,32,-0.000683,0.000000,0.007997,-0.483342
3,OSEBX,0 to 5,32,-0.000372,0.000000,0.012829,-0.164231
4,EQNR,-1 to 1,31,-0.000034,0.000000,0.019450,-0.009767
5,EQNR,0 to 1,32,0.002896,0.000000,0.015669,1.045541
6,EQNR,0 to 3,32,0.009407,0.004684,0.022952,2.318574
7,EQNR,0 to 5,32,0.008811,0.004087,0.021458,2.322775
8,DNB,-1 to 1,31,-0.004960,-0.004639,0.014143,-1.952428
9,DNB,0 to 1,32,-0.001538,0.000000,0.010638,-0.817693
